kernel : Python (Pixi)

In [ ]:
import anndata
import gc
import numpy as np
import os
import pandas as pd
import scanpy as sc
from anndata import AnnData
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir

anndata.settings.allow_write_nullable_strings = True

raw_count_matrix_location = find_env_dir("RAW_COUNT_MATRIX")
h5ad_matrix_location = find_env_dir("H5AD_MATRIX")

def load_mtx(mtx_file: str, col_data_file: str, row_data_file: str, series: str) -> AnnData:
    adata: AnnData = sc.read_mtx(mtx_file).T

    obs_df = pd.read_csv(col_data_file, sep=",", index_col=2)
    var_df = pd.read_csv(row_data_file, sep=",", index_col=0)
    adata.obs = obs_df
    adata.var = var_df

    # Converting to csr_matrix for memory efficiency and speed
    if not isinstance(adata.X, csr_matrix):
        adata.X = csr_matrix(adata.X)

    adata.obs["series"] = series
    adata.obs["batch"] = "not set"
    return adata

#### Main Dataset

In [ ]:
MTX_FILE = "ms_lesions_snRNAseq_cleaned_counts_matrix_2023-09-12.mtx.gz"
COL_DATA_FILE = "ms_lesions_snRNAseq_col_data_2023-09-12.txt.gz"
ROW_DATA_FILE = "ms_lesions_snRNAseq_row_data_2023-09-12.txt.gz"
SERIES = "macnair"
SAVE_FILE_NAME = SERIES + ".h5ad"

data_location = os.path.join(raw_count_matrix_location, "macnair")

adata = load_mtx(
    os.path.join(data_location, MTX_FILE),
    os.path.join(data_location, COL_DATA_FILE),
    os.path.join(data_location, ROW_DATA_FILE),
    series = SERIES
)

In [ ]:

adata = adata[adata.obs["exclude_pseudobulk"] == False].copy()
adata.obs["celltype"] = adata.obs["type_broad"]
adata.obs["celltype_fine"] = adata.obs["type_fine"]
adata.obs["sample"] = adata.obs["sample_id_anon"]
adata.obs["batch"] = adata.obs["sample"]
adata.obs["lesion"] = adata.obs["lesion_type"]
adata.obs["pmi"] = adata.obs["pmi_cat2"]
adata.obs["source"] = adata.obs["sample_source"]
adata.obs["age"] = adata.obs["age_scale"]
adata.obs["sex"] = adata.obs["sex"]
adata.obs["diagnosis"] = adata.obs["diagnosis"]
adata.obs["matter"] = adata.obs["matter"]
adata.obs["pool"] = adata.obs["seq_pool"]
adata.obs["series"] = adata.obs["series"]
keep = ["celltype", "celltype_fine", "sample", "batch", "lesion", "pmi", "source", "age", "sex", "diagnosis", "matter", "pool", "series"]
adata.obs.drop(columns = [c for c in adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [ ]:
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata.write_h5ad(os.path.join(h5ad_matrix_location, "macnair.h5ad"))
del adata
gc.collect()

#### Validation Dataset

In [ ]:
MTX_FILE = "ms_lesions_snRNAseq_validation_cleaned_counts_matrix_2023-09-12.mtx.gz"
COL_DATA_FILE = "ms_lesions_snRNAseq_validation_col_data_2023-09-12.txt.gz"
ROW_DATA_FILE = "ms_lesions_snRNAseq_validation_row_data_2023-09-12.txt.gz"
SERIES = "macnair_validation"
SAVE_FILE_NAME = SERIES + ".h5ad"

data_location = os.path.join(raw_count_matrix_location, "macnair")

adata = load_mtx(
    os.path.join(data_location, MTX_FILE),
    os.path.join(data_location, COL_DATA_FILE),
    os.path.join(data_location, ROW_DATA_FILE),
    series = SERIES
)

In [22]:
adata.obs["source"] = adata.obs["study"]
adata.obs["sample"] = adata.obs["sample_id"]
adata.obs["celltype"] = adata.obs["type_broad"]
adata.obs["pmi"] = adata.obs["pmi"] = pd.cut(
    adata.obs["pmi_minutes"], bins=[-np.inf, 12 * 60, np.inf], labels=["up_tp_12H", "over_12H"] #type: ignore
) 
adata.obs["lesion"] = adata.obs["lesion_type"]
adata.obs["diagnosis"] = adata.obs["diagnosis"]
adata.obs["series"] = adata.obs["series"]
adata.obs["batch"] = adata.obs["batch"]
adata.obs["sex"] = adata.obs["sex"]
keep = ["source", "sample", "celltype", "pmi", "lesion", "diagnosis", "series", "batch", "sex"]
adata.obs.drop(columns = [c for c in adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [ ]:
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata.write_h5ad(os.path.join(h5ad_matrix_location, "macnair_validation.h5ad"))
del adata
gc.collect()